# 전처리

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
sc = StandardScaler()
# 이상치 처리 함수
def handle_outliers(df, num_cols):
    def outlier(col):
        sd = np.std(col)
        m = np.mean(col)
        lower, upper = m - 3*sd, m + 3*sd
        return col.apply(lambda x: upper if x > upper else (lower if x < lower else x))
    for c in num_cols:
        df[c] = outlier(df[c])
    return df

# 인코딩
def encode_labels(df):
    le = LabelEncoder()
    df['영업장명'] = le.fit_transform(df['영업장명'])
    df['영업장명_메뉴명'] = le.fit_transform(df['영업장명_메뉴명'])
    df['메뉴명'] = le.fit_transform(df['메뉴명'])
    return df

## 특징추출1  + 이상치 처리 + 인코딩

In [ ]:
def make_feature1(df, is_train = True):
    df2 = df.copy()
    df2['요일'] = df2['영업일자'].dt.dayofweek
    df2['월'] = df2['영업일자'].dt.month
    df2['일'] = df2['영업일자'].dt.day
    df2['주말'] = (df2['요일'] >= 5).astype(int)
    df2['년도'] = df2['영업일자'].dt.year

    df2['lag_1'] = df2.groupby(['영업장명', '메뉴명'])['매출수량'].shift(1)
    df2['lag_7'] = df2.groupby(['영업장명', '메뉴명'])['매출수량'].shift(7)
    df2['roll_m_7'] = (df2.groupby(['영업장명', '메뉴명'])['매출수량'].shift(1).rolling(window=7).mean())
    df2['roll_s_7'] = (df2.groupby(['영업장명', '메뉴명'])['매출수량'].shift(1).rolling(window=7).std())
    df2['diff_1'] = df2['매출수량'] - df2['lag_1']
    df2['diff_7'] = df2['매출수량'] - df2['lag_7']
    if df2['영업장명'].isin(['느티','카페테리아','포레스트릿','화담']).any():
      df2['월_cos'] = np.cos(2 * np.pi * df2['월'] / 12)
    df2.fillna(0, inplace=True)
    if is_train:
      num_cols = df2.select_dtypes(include = ['int','float'])
      df2 = handle_outliers(df2, num_cols)
      df2 = encode_labels(df2)
      df2.drop('영업일자', axis = 1, inplace = True)
    else:
      df2 = encode_labels(df2)
      df2.drop('영업일자', axis = 1, inplace = True)
    return df2

In [ ]:
train_xl=make_feature1(train)
for i in range(10):
    globals()[f'test_{i:02d}_xl'] = make_feature1(globals()[f'test_{i:02d}'], is_train = False)

# 모델링

## 모델 생성

In [ ]:
# !pip install optuna
# import optuna
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

In [ ]:
target_col = '매출수량'
def create_sliding_window(df, feature_cols, target_col, window_size=28, forecast=7):
    X_list, y_list = [], []

    for store_menu, group in df.groupby(['영업장명', '영업장명_메뉴명']):
        group = group.reset_index(drop=True)
        for i in range(len(group) - window_size - forecast + 1):
            X_window = group.loc[i:i+window_size-1, feature_cols].values.flatten()
            y_window = group.loc[i+window_size:i+window_size+forecast-1, target_col].values
            X_list.append(X_window)
            y_list.append(y_window)

    x = np.array(X_list)
    y = np.array(y_list)
    return x, y

In [ ]:
# ===========================
# XGBoost 모델
# ===========================
def xgb(df, cols):
  xgb_models = {}
  for i,store_name in enumerate(df['영업장명'].unique()):
      print(f"--- XGB :: {train['영업장명'].unique()[i]} 모델 학습 시작 ---")

      store_df = df[df['영업장명'] == store_name]
      x_store, y_store = create_sliding_window(store_df, cols, target_col)

      xgb_base = XGBRegressor(n_estimators=1000,learning_rate=0.2,max_depth=12,gamma=1,reg_lambda=8,reg_alpha=10,random_state=42)
      xgb_model = MultiOutputRegressor(xgb_base)
      weights = np.linspace(1,2,len(y_store)) # 최근 데이터에 가중치

      xgb_model.fit(x_store, y_store, sample_weight = weights)
      xgb_models[store_name] = xgb_model

      print(f"--- XGB :: {train['영업장명'].unique()[i]} 모델 학습 완료 ---")
  return xgb_models

In [ ]:
def tt(df):
  trainxy = pd.DataFrame()
  testxy = pd.DataFrame()
  for i in df['영업장명'].unique():
    df3 = df[df['영업장명'] == i]
    split_idx = int(len(df3) * 0.7)
    trxy = df3.iloc[:split_idx].copy()
    texy  = df3.iloc[split_idx:].copy()
    trainxy = pd.concat([trainxy,trxy], axis = 0)
    testxy = pd.concat([trainxy,trxy], axis = 0)
    cols = trainxy.columns
    cols = [c for c in trainxy.columns if c not in ['영업장명','매출수량']]
  return trainxy, testxy, cols
trainxy, testxy, cols = tt(train_xl)
cols.remove('영업장명_메뉴명')

xgb_models = xgb(trainxy, cols)

--- XGB :: 느티나무 셀프BBQ 모델 학습 시작 ---
--- XGB :: 느티나무 셀프BBQ 모델 학습 완료 ---
--- XGB :: 담하 모델 학습 시작 ---
--- XGB :: 담하 모델 학습 완료 ---
--- XGB :: 라그로타 모델 학습 시작 ---
--- XGB :: 라그로타 모델 학습 완료 ---
--- XGB :: 미라시아 모델 학습 시작 ---
--- XGB :: 미라시아 모델 학습 완료 ---
--- XGB :: 연회장 모델 학습 시작 ---
--- XGB :: 연회장 모델 학습 완료 ---
--- XGB :: 카페테리아 모델 학습 시작 ---
--- XGB :: 카페테리아 모델 학습 완료 ---
--- XGB :: 포레스트릿 모델 학습 시작 ---
--- XGB :: 포레스트릿 모델 학습 완료 ---
--- XGB :: 화담숲주막 모델 학습 시작 ---
--- XGB :: 화담숲주막 모델 학습 완료 ---
--- XGB :: 화담숲카페 모델 학습 시작 ---
--- XGB :: 화담숲카페 모델 학습 완료 ---


In [ ]:
xgb_models

{np.int64(0): MultiOutputRegressor(estimator=XGBRegressor(base_score=None, booster=None,
                                             callbacks=None,
                                             colsample_bylevel=None,
                                             colsample_bynode=None,
                                             colsample_bytree=None, device=None,
                                             early_stopping_rounds=None,
                                             enable_categorical=False,
                                             eval_metric=None,
                                             feature_types=None,
                                             feature_weights=None, gamma=1,
                                             grow_policy=None,
                                             importance_type=None,
                                             interaction_constraints=None,
                                             learning_rate=0.2, max_bin=None,
 

## 점수 확인

In [ ]:
def SMAPE(y_test, y_pred):
    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    a = y_test != 0
    y_test = y_test[a]
    y_pred = y_pred[a]

    return np.mean((np.abs(y_test - y_pred) * 2) / (np.abs(y_test) + np.abs(y_pred)))

def score(tdf, model, cols):
    scores = []
    for i in range(9):
        tdf2 = tdf[tdf['영업장명'] == i]
        xt, yt = create_sliding_window(tdf2, cols, target_col)
        y_pred = model[i].predict(xt)
        y_pred = np.ceil(y_pred)
        scores.append(SMAPE(yt, y_pred))
    s = np.mean(scores)
    return s

In [ ]:
xs = score(testxy,xgb_models, cols)
print('score : ',xs)

score :  0.11419282704783493


# Predict & Submission

In [ ]:
def csw(df, feature_cols, window_size=28):
    X_list = []

    for store in df['영업장명_메뉴명'].unique():
        df2 = df[df['영업장명_메뉴명']==store]

        X_window = df2.loc[:, feature_cols].values.flatten()
        X_list.append(X_window)

    x = np.array(X_list)
    return x

In [ ]:
for i in range(10):
    df = globals()[f'test_{i:02d}_xl']
    start = 1
    for store_name in df['영업장명'].unique():
      df2 = df[df['영업장명'] == store_name]
      if store_name in list(xgb_models.keys()):
          model = xgb_models[store_name]
          x_test = csw(df2, cols)
          pred = model.predict(x_test)
          pred = np.ceil(pred)
          end = start + len(pred)
          sub.iloc[(i*7):(i*7+7), start:end] = np.array(pred.T)
          sub[sub.iloc[:,1:]< 0 ] = 0
          start = end
xsub = sub.copy()

In [ ]:
sample = pd.read_csv('/content/sample_submission.csv')
xsub['영업일자']= sample['영업일자']
xsub.to_csv('xgb.csv', index = False)